# LangChain Demo: Pitti Fashion Exhibitor Analysis

This notebook demonstrates key LangChain capabilities:
1. **Text Loading & Splitting** - Prepare data for embeddings
2. **Vector Embeddings** - Convert text to numerical vectors
3. **Semantic Search** - Find similar items by meaning
4. **RAG (Retrieval-Augmented Generation)** - Answer questions with context
5. **Chain Composition** - Build complex workflows

## Step 1: Load Data

In [ ]:
import pandas as pd
import numpy as np

# Load Pitti exhibitor data
df = pd.read_excel('pitti.xlsx')
print(f"Loaded {len(df)} exhibitors")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nSample:")
df.head(2)

## Step 2: Explore Data

In [ ]:
# Data quality
print("Missing values:")
print(df.isnull().sum())
print(f"\nTotal tags: {df['tag'].nunique()}")
print(f"\nTop categories:")
print(df['tag'].value_counts().head(10))

## Step 3: Text Splitting (LangChain Fundamentals)

Before creating embeddings, we need to split long texts into manageable chunks.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create documents from descriptions
documents = []
for idx, row in df[df['description'].notna()].iterrows():
    doc_text = f"Company: {row['tag']}\n\nDescription: {row['description']}"
    documents.append(doc_text)

print(f"Total documents: {len(documents)}")
print(f"Sample document:\n{documents[0][:300]}...")

In [ ]:
# Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.create_documents(documents)
print(f"Total chunks after splitting: {len(chunks)}")
print(f"\nSample chunk:")
print(chunks[0].page_content)

## Step 4: Create Embeddings (Requires OpenAI API Key)

**Option A: With OpenAI API** (skip to Option B if no API key)

In [ ]:
# OPTION A: Real embeddings with OpenAI
import os

if os.getenv('OPENAI_API_KEY'):
    from langchain_openai import OpenAIEmbeddings
    
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    
    # Test embedding
    test_embedding = embeddings.embed_query("luxury leather shoes")
    print(f"Embedding dimension: {len(test_embedding)}")
    print(f"Sample values: {test_embedding[:5]}")
else:
    print("No OpenAI API key found. Use Option B below for demo mode.")

**Option B: Mock Embeddings (Demo Mode)**

In [ ]:
# OPTION B: Mock embeddings for demo
from langchain_core.embeddings import Embeddings
import hashlib

class MockEmbeddings(Embeddings):
    """Mock embeddings for demo without API key"""
    
    def embed_documents(self, texts):
        return [self._hash_to_embedding(t) for t in texts]
    
    def embed_query(self, text):
        return self._hash_to_embedding(text)
    
    def _hash_to_embedding(self, text):
        """Convert text hash to deterministic embedding"""
        h = hashlib.md5(text.encode()).digest()
        return [float(b) / 255.0 for b in h[:16]]  # 16-dim vector

# Use mock or real embeddings
if os.getenv('OPENAI_API_KEY'):
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    print("Using OpenAI embeddings")
else:
    embeddings = MockEmbeddings()
    print("Using mock embeddings (demo mode)")

## Step 5: Create Vector Store (FAISS)

FAISS (Facebook AI Similarity Search) enables fast similarity searches.

In [ ]:
from langchain_community.vectorstores import FAISS

# Create vector store from chunks
vector_store = FAISS.from_documents(chunks, embeddings)
print(f"Vector store created with {len(chunks)} documents")

## Step 6: Semantic Search Example

In [ ]:
# Search by meaning, not keywords
query = "sustainable and eco-friendly fashion brands"

results = vector_store.similarity_search(query, k=3)

print(f"Query: '{query}'\n")
for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:400] + "..." if len(doc.page_content) > 400 else doc.page_content)

In [ ]:
# Try different queries
queries = [
    "luxury handmade leather goods",
    "vintage retro style clothing",
    "innovative modern design"
]

for query in queries:
    results = vector_store.similarity_search(query, k=1)
    print(f"✓ '{query}'")
    print(f"  → {results[0].page_content.split('Company:')[1].split('Description:')[0].strip()[:50]}")
    print()

## Step 7: RAG (Retrieval-Augmented Generation)

**Only available with OpenAI API key**

In [ ]:
if os.getenv('OPENAI_API_KEY'):
    from langchain_openai import ChatOpenAI
    from langchain.chains import RetrievalQA
    from langchain.prompts import PromptTemplate
    
    # Setup retriever
    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 3}
    )
    
    # Custom prompt
    qa_prompt = PromptTemplate(
        template="""You are an expert on Pitti Immagine fashion fair.
Using the provided exhibitor information, answer the question.
Be specific and reference company names when possible.

Context:
{context}

Question: {question}

Answer:""",
        input_variables=["context", "question"]
    )
    
    # Create QA chain
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        chain_type_kwargs={"prompt": qa_prompt}
    )
    
    print("✓ RAG chain created. Ask a question below:")
else:
    print("⚠️ Skipping RAG demo - requires OPENAI_API_KEY")

In [ ]:
# Ask questions
if os.getenv('OPENAI_API_KEY'):
    question = "Which companies specialize in leather products and when were they founded?"
    
    answer = qa_chain.invoke({"query": question})
    
    print(f"Question: {question}\n")
    print(f"Answer:\n{answer['result']}")

## Step 8: Advanced - Chain Multiple Operations

In [ ]:
if os.getenv('OPENAI_API_KEY'):
    from langchain.chains import LLMChain
    
    # Classify companies
    classify_prompt = PromptTemplate(
        template="""Based on this company description, classify it into fashion categories.
Return 2-3 categories separated by commas.

Description: {description}

Categories:""",
        input_variables=["description"]
    )
    
    classify_chain = LLMChain(llm=llm, prompt=classify_prompt)
    
    # Test on a few companies
    sample_descriptions = df[df['description'].notna()]['description'].head(3)
    
    for i, desc in enumerate(sample_descriptions, 1):
        result = classify_chain.run(description=desc[:200])
        print(f"{i}. {result.strip()}")

## Key LangChain Concepts Demonstrated

| Concept | Example | Use Case |
|---------|---------|----------|
| **Text Splitter** | RecursiveCharacterTextSplitter | Breaking large docs into chunks for embeddings |
| **Embeddings** | OpenAIEmbeddings | Converting text to vectors for semantic search |
| **Vector Store** | FAISS | Storing and searching embeddings efficiently |
| **Retriever** | VectorStore.as_retriever() | Finding relevant documents for a query |
| **Chain** | RetrievalQA | Combining retrieval + LLM for Q&A |
| **Prompt Template** | PromptTemplate | Reusable prompt structure with variables |
| **LLM** | ChatOpenAI | Language model for text generation |

## Next Steps

1. Run `streamlit run pitti_langchain_demo.py` for interactive web UI
2. Experiment with different queries and chunk sizes
3. Try other embeddings models (Hugging Face, Cohere)
4. Implement multi-step agents
5. Add memory for conversational AI